# 06 Window noise, detector calibration, and power

This notebook is self-contained. It downloads NFCorpus and two embedding models from the Hugging Face Hub, encodes all text, and calculates all numbers and figures. It does not read `results/`, `cache/`, or a `silent_rag` module. You need only the pip packages in the next cell. On a laptop CPU, the run time is approximately 10 to 20 minutes.

The benchmark scores **windows** of queries (100 by default). Correlations for each query do not show how a window-level detector operates. This notebook answers the first questions that a detector benchmark needs.

**Questions**
1. How noisy is window-mean nDCG@10 on clean data? What is the false-alarm rate of the "harmful" rule (paired bootstrap, p < 0.05) at window size 100?
2. Are KS and z-score detectors calibrated against a fixed reference window?
3. Do window-level label-free signals track window nDCG (hypothesis H3)? How well do they separate failing queries?
4. How much damage is detectable, and how early? Is the harmful label itself stable?

**Two query populations.** The runner (`src/silent_rag/run.py` with `data.load(split="test")`) uses only the **323 NFCorpus test queries**: a 100-query reference and a 223-query pool. It draws each trial window of 100 from this pool. All 3,237 labelled queries are a different population. A smaller pool decreases the window SD (finite-population correction 0.74, not 0.98). It also causes much overlap between trial windows (approximately 45 shared queries between two windows, not approximately 3). Thus, each window-level section runs on two populations:
- **`runner` (primary):** the test queries, with the reference and pool selected as `run_dataset` selects them (`default_rng(seed).permutation`, first 100 = reference).
- **`all` (sensitivity):** all 3,237 labelled queries.

**Method**
- Random clean windows, compared with the finite-population variance formula.
- A sign-flip null for the harmful rule.
- False-alarm rates over 200 reference draws.
- Window-level correlation with a permutation null.
- Power and earliness under a synthetic degradation (a fraction of the queries return random documents).
- A Monte Carlo check of how frequently the harmful call depends on the bootstrap seed.
- Minimum detectable effect.

**These are not the failure injectors of the project.** The synthetic degradation is a placeholder with a known effect size. Its purpose is to check the statistics. It is not F1 to F5.

**Scope.** This notebook uses only NFCorpus. NFCorpus is the development set of the project. SciFact and FiQA are held-out sets for the final run. This notebook does not load them.

In [ ]:
# On a new environment, remove the "#" from the next line.
# %pip install -q datasets==5.1.0 sentence-transformers==6.1.0 torch==2.14.1 scikit-learn==1.9.1 scipy==1.18.1 pandas==3.0.6 numpy==2.5.4 matplotlib=3.11.2

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # On macOS, this prevents an OpenMP conflict. It has no effect on other systems.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy import stats
from scipy.stats import ks_2samp
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

SEED = 0
np.random.seed(SEED)
K = 10
W = 100                                     # Window size
SPLITS = ["train", "validation", "test"]    # The Hub name of the qrels split is "validation", not "dev".
MAX_SEQ = 512

NFCORPUS = dict(
    repo="BeIR/nfcorpus", revision="b5026a0e96e8a7ac4f95f482a596389289d46269",
    qrels="BeIR/nfcorpus-qrels", qrels_revision="a451b3b26d3ae1358f259c1a3a4dd61fcea35a65",
)
MODELS = {
    "multi-qa-MiniLM": dict(repo="sentence-transformers/multi-qa-MiniLM-L6-cos-v1", revision="b207367332321f8e44f96e224ef15bc607f4dbf0"),
    "all-MiniLM": dict(repo="sentence-transformers/all-MiniLM-L6-v2", revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41"),
}

## Helpers, data, and clean retrieval

This section defines the functions that load the data, encode the text, do an exact cosine search, and calculate nDCG@10. nDCG@10 uses graded gain and a log2 discount, as in BEIR. It also defines the paired bootstrap for the harmful rule. For clean and damaged scores of each query, the bootstrap gives the mean drop `clean - damaged` and a one-sided p-value for "no drop" (the share of bootstrap means <= 0). `paired_bootstrap` is a copy of `src/silent_rag/eval/ground_truth.py:paired_bootstrap`, with the same defaults (`n_boot = 2000`). Thus, the calibration here is the calibration of the rule that the runner uses. The notebook uses all labelled queries that have one or more relevant documents.

**Populations.** `ref_pool(pop, ref_i)` returns a reference window and the disjoint pool. For `ref_i = 0`, it gives the same split as `run_dataset` (`np.random.default_rng(seed).permutation(n)`, first `W` = reference). Other `ref_i` values draw more references from the same population, to measure the variation across references. The runner keeps all test queries that have qrels. This notebook keeps the test queries with one or more positive judgments. The cell prints how many test queries this removes. If it removes none, the positions are the same as in the runner.

**The tie rule.** The notebook breaks exact score ties by document index (stable sort). `pipeline.Retriever` uses FAISS `IndexFlatIP`, which can break ties in a different order for the queries with a tie at rank 10. Thus, the mean nDCG for each query can be different from the clean baseline of the runner in the fourth decimal.

In [ ]:
def doc_text(title, text):
    return f"{(title or '').strip()} {(text or '').strip()}".strip()


def load_nfcorpus():
    c = NFCORPUS
    corpus = load_dataset(c["repo"], "corpus", split="corpus", revision=c["revision"])
    queries = load_dataset(c["repo"], "queries", split="queries", revision=c["revision"])
    qtext = {str(q["_id"]): q["text"] for q in queries}
    split_of, qrels = {}, {}
    for split in SPLITS:
        for row in load_dataset(c["qrels"], split=split, revision=c["qrels_revision"]):
            q = str(row["query-id"])
            if split_of.setdefault(q, split) == split:
                qrels.setdefault(q, {})[str(row["corpus-id"])] = int(row["score"])
    doc_ids = [str(d["_id"]) for d in corpus]
    docs = [doc_text(t, x) for t, x in zip(corpus["title"], corpus["text"])]
    return doc_ids, docs, {q: qtext[q] for q in qrels}, qrels, split_of


def ndcg_at_k(ranked, rels, k=K):
    dcg = sum(rels.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted(rels.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def encode(enc, texts, batch_size=32):
    return np.ascontiguousarray(enc.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)


def search(D, Q, k=K):
    S = Q @ D.T
    # Stable full sort: if two scores are equal (the corpus has duplicate documents), the lower document index comes first.
    # Thus, the top-k does not depend on the search routine.
    top = np.argsort(-S, axis=1, kind="stable")[:, :k]
    return top, np.take_along_axis(S, top, axis=1)


def paired_bootstrap(clean, damaged, n_boot=2000, seed=0):
    # The same as src/silent_rag/eval/ground_truth.py:paired_bootstrap
    rng = np.random.default_rng(seed)
    diff = clean - damaged
    boots = diff[rng.integers(0, len(diff), size=(n_boot, len(diff)))].mean(axis=1)
    return float(diff.mean()), float((boots <= 0).mean())


doc_ids, docs, queries, qrels, split_of = load_nfcorpus()
qids = [q for q in queries if any(r > 0 for r in qrels[q].values())]
rel = [{d for d, v in qrels[q].items() if v > 0} for q in qids]
N = len(qids)


def run_model(label):
    spec = MODELS[label]
    enc = SentenceTransformer(spec["repo"], revision=spec["revision"], device="cpu")
    enc.max_seq_length = MAX_SEQ
    D = encode(enc, docs)
    Q = encode(enc, [queries[q] for q in qids])
    top, scores = search(D, Q)
    ids = [[doc_ids[j] for j in row] for row in top]
    ndcg = np.array([ndcg_at_k(r, qrels[q]) for r, q in zip(ids, qids)])
    recall = np.array([len(r & set(i)) / len(r) for r, i in zip(rel, ids)])
    return {"D": D, "Q": Q, "scores": scores, "ndcg": ndcg, "recall": recall}


r0 = run_model("multi-qa-MiniLM")
alt = run_model("all-MiniLM")
sc = r0["scores"]
feat = pd.DataFrame({"top1": sc[:, 0], "top10_mean": sc.mean(1), "gap": sc[:, 0] - sc[:, 1]})
assert len(feat) == len(r0["ndcg"]) == len(alt["ndcg"]) == N   # All arrays have the same query order
print(N, "queries | mean nDCG@10", round(float(r0["ndcg"].mean()), 4), "| failure rate (recall@10 = 0)", round(float((r0["recall"] == 0).mean()), 3))

# Query populations. "runner" is the same as src/silent_rag/run.py: data.load(split="test") keeps the test queries in qrels
# order (the same relative order as here). Then run_dataset permutes them with default_rng(seed): first W = reference.
test_pos = np.array([i for i, q in enumerate(qids) if split_of[q] == "test"])
n_test_with_qrels = sum(split_of[q] == "test" for q in queries)
POPS = {"runner": test_pos, "all": np.arange(N)}


def ref_pool(pop, ref_i, seed=SEED):
    base = POPS[pop]
    perm = np.random.default_rng(seed if ref_i == 0 else [seed, ref_i]).permutation(len(base))
    return base[perm[:W]], base[perm[W:]]


rows = []
for pop, base in POPS.items():
    n_pool = len(base) - W
    rows.append({"population": pop, "queries": len(base), "reference": W, "pool": n_pool,
                 "fpc_sd_factor": np.sqrt((n_pool - W) / (n_pool - 1)), "expected_overlap_between_two_windows": W * W / n_pool,
                 "mean_ndcg": r0["ndcg"][base].mean()})
print("test queries with qrels:", n_test_with_qrels, "| with a positive judgment (used here):", len(test_pos))
pd.DataFrame(rows).round(4)

## 1. Query-level AUROC for failing queries

A query fails when recall@10 = 0. The AUROC is for the label-free score as a predictor of failure. A lower score means that a failure is more probable, so the notebook uses the negative of the score. A cross-validated logistic regression on the three scores shows the gain from a combination of the scores. AUROC = 0.5 is chance. The table shows both populations.

**Why this test**
- **Estimand:** the probability that a random failing query has a lower top-1 (or other) score than a random successful query.
- **Unit of independence:** the query (bootstrap over queries). This is optimistic, because queries share topics and relevant documents. Thus, the CI is a lower limit on the uncertainty.
- **Assumptions:** no assumption about the score distribution (AUROC uses ranks). The percentile bootstrap assumes that the sampling distribution of the AUROC is approximately symmetric. This is true when the AUROC is not near 0 or 1.
- **Why not the usual alternative:** a Mann-Whitney p-value only shows that the AUROC is different from 0.5. This is not in doubt. The important quantity is the size of the separation.
- **Selection caveat:** if the project selects top-1 as the detector signal because of its correlation with nDCG on these queries, all top-1 values here are in-sample.

In [ ]:
def auc_ci(y, s, B=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    a = roc_auc_score(y, s)
    boot = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        boot.append(roc_auc_score(y[i], s[i]))
    return a, *np.percentile(boot, [2.5, 97.5])


y_fail_all = (r0["recall"] == 0).astype(int)
rows = []
for pop, base in POPS.items():
    y_fail = y_fail_all[base]
    for f in feat:
        a, lo, hi = auc_ci(y_fail, -feat[f].to_numpy()[base])
        rows.append({"population": pop, "signal": f, "auroc_for_failure": a, "ci_lo": lo, "ci_hi": hi, "n": len(y_fail), "failure_rate": y_fail.mean()})
    p = cross_val_predict(LogisticRegression(max_iter=1000), feat.to_numpy()[base], y_fail,
                          cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="decision_function")
    a, lo, hi = auc_ci(y_fail, p)
    rows.append({"population": pop, "signal": "logistic(top1, top10_mean, gap), 5-fold CV", "auroc_for_failure": a, "ci_lo": lo, "ci_hi": hi,
                 "n": len(y_fail), "failure_rate": y_fail.mean()})
pd.DataFrame(rows).round(3)

## 2. Noise floor of window-mean nDCG@10

**Model.** A window is a simple random sample of `w` queries without replacement from the **pool** (the population minus the reference, as in the runner). Its mean has the variance `sigma^2 / w * (P - w) / (P - 1)` (finite-population correction). `P` is the pool size, and `sigma` is the SD of nDCG for each query over the pool.

**Assumption tested.** The empirical SD of 3,000 random windows agrees with this formula. The window means are approximately normal (skewness, and a Shapiro-Wilk test on 500 windows). z-thresholds on window means need this. nDCG is bounded and zero-inflated, so normality is not guaranteed for small windows.

**Why this test**
- **Estimand:** the SD of the mean nDCG@10 of a clean window. This is the noise that each window-level rule must see through.
- **Unit of independence:** the window, drawn independently from a fixed pool. Windows from the same pool share queries. The finite-population term includes this.
- **Assumptions:** simple random sampling. Real traffic has topic clusters, and this model does not include them (see Findings).
- **Why not the usual alternative:** the iid formula `sigma / sqrt(w)` gives a noise that is too large when the pool is small. With the 223-query pool of the runner, a 100-query window is 45% of the pool.
- **Resolution:** with 3,000 windows, the empirical SD has a relative Monte Carlo error of approximately 1.3%. Shapiro-Wilk on 500 windows finds only clear departures from normality.

In [ ]:
rows = []
for pop in POPS:
    _, pool = ref_pool(pop, 0)
    nd = r0["ndcg"][pool]
    P = len(nd)
    sigma = nd.std(ddof=0)
    rng = np.random.default_rng([SEED, 2])
    for w in (25, 50, 100, 200, 400):
        if w >= P:
            continue
        means = np.array([nd[rng.choice(P, w, replace=False)].mean() for _ in range(3000)])
        theory = sigma / np.sqrt(w) * np.sqrt((P - w) / (P - 1))
        rows.append({"population": pop, "w": w, "pool": P, "empirical_sd": means.std(ddof=1), "theory_sd": theory,
                     "iid_sd_no_fpc": sigma / np.sqrt(w), "ratio": means.std(ddof=1) / theory,
                     "skew": stats.skew(means), "shapiro_p": stats.shapiro(means[:500]).pvalue,
                     "p2.5": np.percentile(means, 2.5), "p97.5": np.percentile(means, 97.5)})
pd.DataFrame(rows).round(4)

## 3. Calibration of the `harmful` rule at w = 100

The benchmark calls a condition harmful when `paired_bootstrap(clean, damaged)` gives p < 0.05. If clean and damaged are exchangeable (no real damage), the false-positive rate must be approximately 5%. To test this with real nDCG for each query, the notebook builds a **sign-flip null** from the differences `d_i` between the baseline and `all-MiniLM` for each query. The "damaged" nDCG of a window is `clean_i - s_i * d_i`, with random signs `s_i = +/-1`. By construction, this has a symmetric paired difference with a mean of zero. But it keeps the real, skewed, zero-inflated variance of nDCG. The test uses 1,000 windows of 100 from the pool of each population, with 2,000 bootstrap resamples each (the setting of the runner).

**Why this test**
- **Estimand:** the false-positive rate of `harmful = p < 0.05` when there is no true drop.
- **Unit of independence:** the window. In a window, the rule resamples queries with replacement.
- **Assumptions:** the sign-flip null is valid when the paired difference is symmetric about zero under "no damage". Real injected damage is one-sided. Thus, this checks the procedure, not the benchmark.
- **Why not the usual alternative:** a paired t-test assumes an approximately normal mean difference. nDCG is zero-inflated and bounded, and the window has 100 queries. The runner uses the bootstrap rule, so the calibration of this rule is the important value. A comparison of the rule with itself on the same clean data gives no information, because all differences are exactly zero (p = 1).
- **Resolution:** 1,000 windows give a binomial SE of approximately 0.7 percentage points at 5%. The table gives the Jeffreys interval.

In [ ]:
d_model = r0["ndcg"] - alt["ndcg"]
print("per-query difference SD between two clean models:", round(float(d_model.std()), 4), "| mean:", round(float(d_model.mean()), 4))

rows, pv_store = [], {}
for pop in POPS:
    _, pool = ref_pool(pop, 0)
    rng = np.random.default_rng([SEED, 3])
    pvals = []
    for j in range(1000):
        idx = rng.choice(pool, W, replace=False)
        s = rng.choice([-1.0, 1.0], W)
        clean = r0["ndcg"][idx]
        pvals.append(paired_bootstrap(clean, clean - s * d_model[idx], n_boot=2000, seed=j)[1])
    pvals = np.array(pvals)
    k_ = int((pvals < 0.05).sum())
    lo, hi = stats.beta.ppf([0.025, 0.975], k_ + 0.5, len(pvals) - k_ + 0.5)
    rows.append({"population": pop, "windows": len(pvals), "false_positive_rate": k_ / len(pvals), "jeffreys_lo": lo, "jeffreys_hi": hi,
                 "uniformity_ks_p": stats.kstest(pvals, "uniform").pvalue})
    pv_store[pop] = pvals
display(pd.DataFrame(rows).round(4))

fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for ax, (pop, pvals) in zip(axes, pv_store.items()):
    ax.hist(pvals, bins=20, color="lightgrey")
    ax.set_title(pop, fontsize=9)
    ax.set_xlabel("paired_bootstrap p under the sign-flip null")
axes[0].set_ylabel("windows")
fig.tight_layout()
plt.show()

## 4. Calibration of KS and z detectors against a fixed reference window

Detectors compare a scoring window with a **reference window** that the notebook draws one time from the clean system. The detectors use two label-free window statistics: the two-sample **KS** test, and the **z-score of the window mean** against the reference. Each statistic uses the top-1 score and the top-1 gap. The reference is disjoint from all scoring windows. The notebook measures the rate over 200 reference draws for each population, with 300 clean windows for each reference. This gives the *conditional* false-alarm rate for a given reference (the rate that one benchmark run has).

**The z detector uses the Welch standard error.** A naive standard error `sd_ref / sqrt(w)` ignores that the reference *mean* also comes from only 100 queries. With equal sizes, the expected false-alarm rate at 1.96 is then `P(|Z| > 1.96 / sqrt 2) = 16.6%`. Thus, the detector uses the **Welch** standard error `sqrt(var_ref / n_ref + var_win / w)`. This includes the error of both means, and it does not assume that the damaged window has the variance of the reference (damage changes the variance). The table keeps the naive form as a labelled `z naive SE (bug)` row to show the size of the error.

**Why this test**
- **Estimand:** the false-alarm rate of each rule on clean windows for a given reference, and how much it changes across references.
- **Unit of independence:** the reference (200 draws). The 300 windows for each reference give the conditional rate of that reference.
- **Assumptions:** KS assumes continuous data. The top-1 gap has exact ties (gap = 0 for duplicate documents), so KS is a little conservative for the gap. The z rule assumes that the difference of means is approximately normal (section 2 supports this at w = 100).
- **Why not the usual alternative:** an unconditional rate (all references pooled) does not show that one run has only one reference. The spread across references is the practically important value.
- **Resolution:** each conditional rate has a binomial SE of approximately 1.3 points at 5% (300 windows).

In [ ]:
def window_stats(feat_ref, feat_win):
    """Return the KS statistic, KS p, Welch z, and the naive z (reference SD only; the reference mean is treated as known)."""
    ks = ks_2samp(feat_ref, feat_win)
    diff = feat_win.mean() - feat_ref.mean()
    se = np.sqrt(feat_ref.var(ddof=1) / len(feat_ref) + feat_win.var(ddof=1) / len(feat_win))
    z_naive = diff / (feat_ref.std(ddof=1) / np.sqrt(len(feat_win)) + 1e-12)
    return np.array([ks.statistic, ks.pvalue, diff / (se + 1e-12), z_naive])


cal_rows = []
for pop in POPS:
    for sig in ("top1", "gap"):
        x = feat[sig].to_numpy()
        rng = np.random.default_rng([SEED, 4])
        cond = {"KS p<0.05": [], "|z|>1.96 Welch SE": [], "|z|>1.96 naive SE (bug)": []}
        for ref_i in range(200):
            ref, pool = ref_pool(pop, ref_i)
            st = np.array([window_stats(x[ref], x[rng.choice(pool, W, replace=False)]) for _ in range(300)])
            cond["KS p<0.05"].append((st[:, 1] < 0.05).mean())
            cond["|z|>1.96 Welch SE"].append((np.abs(st[:, 2]) > 1.96).mean())
            cond["|z|>1.96 naive SE (bug)"].append((np.abs(st[:, 3]) > 1.96).mean())
        for rule, v in cond.items():
            v = np.array(v)
            cal_rows.append({"population": pop, "signal": sig, "rule": rule, "mean_false_alarm": v.mean(), "sd_across_references": v.std(ddof=1),
                             "p5": np.percentile(v, 5), "p95": np.percentile(v, 95), "max": v.max(), "runner_reference_0": v[0]})
pd.DataFrame(cal_rows).round(4)

## 5. Window-level tracking (hypothesis H3)

This section divides the queries into disjoint windows of `w`. It calculates the window-mean nDCG and the window-mean signal, and it correlates them across windows (Spearman). The statistic is the mean of this correlation over **200 random partitions**. Windows from different partitions share queries, so they are not independent. Thus, the inference uses a **permutation null**: shuffle the signal across queries (this breaks its link to nDCG), and calculate **the same statistic** (the mean over 200 partitions) 300 times. Window sizes 50, 100, and 200 show how the signal becomes stronger with aggregation.

**Population.** H3 runs only on **all** labelled queries. The 323 test queries of the runner give only three windows of 100, which is too few for a correlation across windows.

**H3 (confirmatory):** at w = 100, the window-mean top-1 score has a positive correlation with the window-mean nDCG@10 (one-sided). This notebook tests H3 alone. It is not in the Holm family of H1 and H2.

**Why this test**
- **Estimand:** the expected Spearman correlation between the window-mean signal and the window-mean nDCG, for a random partition into windows of `w`.
- **Unit of independence:** the query (the permutation shuffles the signal values of the queries).
- **Assumptions:** under the null of no association, the signal is exchangeable across queries.
- **Why not the usual alternative:** the p-value of the Spearman for one partition uses only `N / w` windows, and it depends on the partition. The mean over partitions removes this dependence. The permutation null keeps the test exact for the mean statistic.
- **Resolution:** the lowest possible p is 1/301 = 0.0033. The notebook reports it as `p < 0.004` when the observed value is larger than all null values.

In [ ]:
def partition_corr(x, y, w, parts, rng):
    nw = len(x) // w
    out = np.empty(parts)
    for p_ in range(parts):
        idx = rng.permutation(len(x))[: nw * w].reshape(nw, w)
        out[p_] = stats.spearmanr(x[idx].mean(1), y[idx].mean(1))[0]
    return out


PARTS, NULL_REPS = 200, 300
y = r0["ndcg"]
rows, store = [], {}
for sig in ("top1", "top10_mean", "gap"):
    x = feat[sig].to_numpy()
    q_rho = stats.spearmanr(x, y)[0]
    for w in (50, 100, 200):
        rng = np.random.default_rng([SEED, 5])
        obs = partition_corr(x, y, w, PARTS, rng).mean()
        null = np.array([partition_corr(rng.permutation(x), y, w, PARTS, rng).mean() for _ in range(NULL_REPS)])  # The same statistic
        idx = np.random.default_rng(SEED).permutation(len(y))[: (len(y) // w) * w].reshape(-1, w)
        slope = stats.linregress(x[idx].mean(1), y[idx].mean(1)).slope
        rows.append({"signal": sig, "w": w, "n_windows": len(y) // w, "query_level_rho": q_rho, "window_level_rho_mean": obs,
                     "null_mean": null.mean(), "null_sd": null.std(ddof=1), "null_max": null.max(),
                     "perm_p_one_sided": (1 + (null >= obs).sum()) / (1 + len(null)), "slope_ndcg_per_unit_signal": slope})
        store[(sig, w)] = (obs, null)
display(pd.DataFrame(rows).round(4))

obs, null = store[("top1", 100)]
fig, ax = plt.subplots(figsize=(4, 2.8))
ax.hist(null, bins=25, color="lightgrey")
ax.axvline(obs, color="tab:red")
ax.set_title("top-1 score, w=100 (all queries)", fontsize=9)
ax.set_xlabel("mean window-level Spearman")
fig.tight_layout()
plt.show()

## 6. Power and earliness under a synthetic degradation

**Synthetic degradation (not F1 to F5).** In a damaged window, a fraction `s` (the severity) of the queries return **random documents** and not their real results. For these queries, the top-1 score becomes the best cosine to 10 random documents, and the notebook calculates nDCG@10 again on the random list. This gives a known, label-based effect to test the statistics.

**Design (the same as the runner).**
- *Paired across severities.* For each reference and trial `m`, the notebook draws the window, the order of damage, and the random documents one time, from `default_rng([SEED, ref_i, m])`. It uses them again at each severity. Severity `s` damages the first `round(s * 100)` queries of that order. Thus, the damage is **nested**: a higher severity damages a superset of the queries. The runner also scores the same window for each condition of trial `t`. This design keeps window noise out of the severity effect.
- *Severity grid* `0, 0.05, 0.1, 0.2, ..., 0.9`. This includes the 0.1 to 0.9 range of the project.
- *Harmful rule* with `n_boot = 2000` and `seed = m`, as `run.py` calls it (`seed=trial`). At severity 0, the damaged window is equal to the clean window, so its rate is 0 by construction. Section 3 gives its false-positive rate under the sign-flip null.
- *30 references* for each population. Reference 0 of `runner` is the actual reference of the runner.

**Rules (on the top-1 score).**
- **Nominal:** KS p < 0.05 and Welch |z| > 1.96. A user selects these without calibration. The naive-SE z is a labelled bug row.
- **Oracle-calibrated:** the threshold is the 95th percentile of the statistic on 300 clean windows from the pool. This is an **upper limit**. A `Detector` sees only `fit(reference)`, so it cannot draw clean windows from the pool.
- **Reference-only calibration:** the threshold is the 95th percentile of the statistic between the reference and 300 bootstrap resamples of the reference. A detector *can* calculate this in `fit()`. Its rate at severity 0 shows whether it is calibrated.

**Earliness** is the smallest severity `s` at which the detection rate is 80% or more at `s` **and at all higher severities** (for each reference). Thus, one noisy early crossing does not count.

**Effective damage.** A damaged query whose clean nDCG is already 0 cannot lose nDCG. Approximately one third of queries have recall@10 = 0 on the clean system. Thus, at severity `s`, only approximately two thirds of the damaged queries can have a real drop.

**Why this test**
- **Estimand:** the detection probability of each rule at each severity, for a reference that the notebook draws as the runner draws it, and its spread across references.
- **Unit of independence:** the reference (30 draws). For each reference, 200 trials give the conditional rate (binomial SE of 3.5 points or less).
- **Assumptions:** only the synthetic damage model. The detection rates are empirical.
- **Why not the usual alternative:** an analytic power formula needs normal, symmetric differences. Zero-inflated nDCG with all-or-nothing damage does not give this (section 7 compares them).
- **Error bars:** the markers are the mean over references. The bars are the 5th to 95th percentiles across references. This is the variation that one run has. No lines connect the markers, because there is no information between grid points.

In [ ]:
D0, Q0 = r0["D"], r0["Q"]
clean_top1, clean_nd = feat["top1"].to_numpy(), r0["ndcg"]


def random_lists(win, rng, n_docs=10):
    """Return the top-1 score and nDCG@10 if each query in `win` returns 10 random documents (a new draw for each call)."""
    t1, nd_r = np.empty(len(win)), np.empty(len(win))
    for j, i in enumerate(win):
        pick = rng.choice(len(D0), n_docs, replace=False)
        t1[j] = (D0[pick] @ Q0[i]).max()
        nd_r[j] = ndcg_at_k([doc_ids[p] for p in pick], qrels[qids[i]])
    return t1, nd_r


def p95_thresholds(st):
    return np.percentile(st[:, 0], 95), np.percentile(np.abs(st[:, 2]), 95)


sev_grid = (0.0, 0.05) + tuple(np.round(np.arange(0.1, 1.0, 0.1), 1))
N_REF, M_DAM, M_CLEAN, M_REFBOOT = 30, 200, 300, 300
RULES = ["harmful_rate", "ks_nominal", "z_nominal", "z_naive_bug", "ks_oracle", "z_oracle", "ks_refonly", "z_refonly"]

rows, trials_store = [], {}
for pop in POPS:
    for ref_i in range(N_REF):
        ref, pool = ref_pool(pop, ref_i)
        xr = clean_top1[ref]
        rng = np.random.default_rng([SEED, 6, ref_i])
        cl = np.array([window_stats(xr, clean_top1[rng.choice(pool, W, replace=False)]) for _ in range(M_CLEAN)])
        rb = np.array([window_stats(xr, xr[rng.integers(0, W, W)]) for _ in range(M_REFBOOT)])
        (ks_or, z_or), (ks_rf, z_rf) = p95_thresholds(cl), p95_thresholds(rb)
        trials = []
        for m in range(M_DAM):
            rt = np.random.default_rng([SEED, ref_i, m])
            win = rt.choice(pool, W, replace=False)
            trials.append((win, rt.permutation(W)) + random_lists(win, rt))
        if ref_i == 0:
            trials_store[pop] = trials
        for s in sev_grid:
            k = int(round(s * W))
            hits = dict.fromkeys(RULES, 0)
            dam = []
            for m, (win, order, rt1, rnd) in enumerate(trials):
                t1, nd_w = clean_top1[win].copy(), clean_nd[win].copy()
                t1[order[:k]], nd_w[order[:k]] = rt1[order[:k]], rnd[order[:k]]
                _, p_h = paired_bootstrap(clean_nd[win], nd_w, n_boot=2000, seed=m)   # The same as run.py: seed=trial
                ks_stat, ks_p, z, z_naive = window_stats(xr, t1)
                for rule, fired in (("harmful_rate", p_h < 0.05), ("ks_nominal", ks_p < 0.05), ("z_nominal", abs(z) > 1.96),
                                    ("z_naive_bug", abs(z_naive) > 1.96), ("ks_oracle", ks_stat > ks_or), ("z_oracle", abs(z) > z_or),
                                    ("ks_refonly", ks_stat > ks_rf), ("z_refonly", abs(z) > z_rf)):
                    hits[rule] += fired
                dam.append((ks_stat, abs(z)))
            dam = np.array(dam)
            yy = np.r_[np.ones(M_DAM), np.zeros(M_CLEAN)]
            aucs = [roc_auc_score(yy, np.r_[dam[:, 0], cl[:, 0]]), roc_auc_score(yy, np.r_[dam[:, 1], np.abs(cl[:, 2])])] if s > 0 else [np.nan, np.nan]
            rows.append({"population": pop, "reference": ref_i, "severity": s, **{r_: v / M_DAM for r_, v in hits.items()},
                         "auroc_ks": aucs[0], "auroc_z": aucs[1]})
by_ref = pd.DataFrame(rows)
cols = RULES + ["auroc_ks", "auroc_z"]
g = by_ref.groupby(["population", "severity"])[cols]
power = g.mean().round(3)
spread = (g.quantile(0.95) - g.quantile(0.05)).round(3)
print("mean detection rate over 30 references")
display(power)
print("width of the 5th-95th percentile range across references")
display(spread)
print("runner's own reference (reference 0, population 'runner')")
by_ref[(by_ref["population"] == "runner") & (by_ref["reference"] == 0)].set_index("severity")[cols].round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=True, sharex=True)
panels = (("nominal thresholds", ["harmful_rate", "ks_nominal", "z_nominal"]),
          ("calibrated: oracle (solid) / reference-only (open)", ["harmful_rate", "ks_oracle", "z_oracle", "ks_refonly", "z_refonly"]))
offsets = dict(zip(RULES, np.linspace(-0.012, 0.012, len(RULES))))
for row_axes, pop in zip(axes, POPS):
    d = by_ref[by_ref["population"] == pop]
    gm = d.groupby("severity")[RULES]
    mean, lo, hi = gm.mean(), gm.quantile(0.05), gm.quantile(0.95)
    for ax, (title, series) in zip(row_axes, panels):
        for col in series:
            xs = mean.index + offsets[col]
            yerr = np.clip([mean[col] - lo[col], hi[col] - mean[col]], 0, None)   # Clip float round-off at 0
            ax.errorbar(xs, mean[col], yerr=yerr, fmt="o", ms=4, capsize=2, ls="none",
                        mfc="none" if col.endswith("refonly") else None, label=col)
        ax.axhline(0.8, color="grey", ls=":")
        ax.axhline(0.05, color="grey", ls=":")
        ax.set_title(f"{pop}: {title}", fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("fraction of queries degraded (markers only; no interpolation)")
for ax in axes[:, 0]:
    ax.set_ylabel("detection rate (mean, 5-95% over refs)")
axes[0, 0].legend(fontsize=7)
axes[0, 1].legend(fontsize=7)
fig.tight_layout()
plt.show()


def earliest_sustained(d, col, level=0.8):
    d = d[d["severity"] > 0].sort_values("severity")
    ok = (d[col].to_numpy() >= level)[::-1].cumprod()[::-1].astype(bool)   # True where this severity and all higher severities reach the level
    return float(d["severity"].to_numpy()[ok][0]) if ok.any() else np.inf


rows = []
for pop in POPS:
    for col in RULES:
        e = np.array([earliest_sustained(d, col) for _, d in by_ref[by_ref["population"] == pop].groupby("reference")])
        rows.append({"population": pop, "rule": col, "earliest_80pct_min": e.min(), "earliest_80pct_median": np.median(e),
                     "earliest_80pct_max": e.max(), "refs_never_reaching_80pct": int(np.isinf(e).sum())})
pd.DataFrame(rows)

## 6b. Does the harmful call depend on the bootstrap seed?

`harmful = p < 0.05` is a Monte Carlo decision. `p` comes from `n_boot` resamples. Thus, a window with a true bootstrap p near 0.05 can be harmful under one seed and not harmful under a different seed. The `trials` of the runner are reproducible only because the seed is fixed (`seed=trial`). This section uses the reference-0 trials of the runner population at severities 0.05 and 0.1 (where windows are near the threshold). It calculates the call again under 30 bootstrap seeds. It reports the share of windows whose call is **not the same** across seeds, for `n_boot` = 500, 2,000 (the setting of the runner), and 10,000.

**Why this test**
- **Estimand:** the probability that the harmful call of a window changes because of the bootstrap seed alone.
- **Unit of independence:** the window (100 for each severity).
- **Assumptions:** none. This measures the Monte Carlo error directly.
- **Why not the usual alternative:** the analytic SE of `p` (`sqrt(p(1 - p) / n_boot)`, approximately 0.005 at `n_boot = 2000`) shows how far `p` moves. It does not show how many benchmark rows change. A count of the changed calls shows this.

In [ ]:
rows = []
trials = trials_store["runner"][:100]
for s in (0.05, 0.1):
    k = int(round(s * W))
    for nb in (500, 2000, 10000):
        flips, base_rate = 0, []
        for win, order, rt1, rnd in trials:
            nd_w = clean_nd[win].copy()
            nd_w[order[:k]] = rnd[order[:k]]
            calls = np.array([paired_bootstrap(clean_nd[win], nd_w, n_boot=nb, seed=b)[1] < 0.05 for b in range(30)])
            flips += calls.min() != calls.max()
            base_rate.append(calls.mean())
        rows.append({"population": "runner", "severity": s, "n_boot": nb, "windows": len(trials),
                     "share_windows_call_depends_on_seed": flips / len(trials), "mean_harmful_rate": np.mean(base_rate)})
pd.DataFrame(rows).round(3)

In [ ]:
# Why more resamples do not decrease the seed dependence: with sparse damage, the bootstrap p is on a lattice.
# Only damaged queries with a clean nDCG > 0 change (a failing query has nothing to lose). If j of these queries
# have a positive drop and none has a negative drop, the mean drop of a resample is 0 when the resample misses all j.
# Thus, p is approximately (1 - j/W)^W. For j = 3, this is 0.0476, just below 0.05. Then Monte Carlo noise decides the call.
lattice = pd.DataFrame({"effective_damaged_queries_j": range(1, 7), "approx_p": [(1 - j / W) ** W for j in range(1, 7)]})
display(lattice.round(4))

rows = []
k = int(round(0.05 * W))
for win, order, rt1, rnd in trials:
    nd_w = clean_nd[win].copy()
    nd_w[order[:k]] = rnd[order[:k]]
    d = clean_nd[win] - nd_w
    calls = np.array([paired_bootstrap(clean_nd[win], nd_w, n_boot=2000, seed=b)[1] < 0.05 for b in range(30)])
    rows.append({"j_positive_drop": int((d > 0).sum()), "any_negative_drop": bool((d < 0).any()),
                 "call_depends_on_seed": calls.min() != calls.max(), "harmful_share_over_seeds": calls.mean()})
lat = pd.DataFrame(rows)
lat.groupby("j_positive_drop").agg(windows=("call_depends_on_seed", "size"), share_seed_dependent=("call_depends_on_seed", "mean"),
                                   mean_harmful_share=("harmful_share_over_seeds", "mean"), any_negative=("any_negative_drop", "mean")).round(3)

## 7. Minimum detectable nDCG drop

For a paired window test at `alpha = 0.05` (one-sided) and 80% power, the minimum detectable mean drop is `(z_0.95 + z_0.80) * sd_d / sqrt(w)`. `sd_d` is the SD of the difference in nDCG between clean and damaged for each query. `sd_d` depends on the failure, and the failure is unknown until injection. Thus, the first table is a sensitivity grid. For the small pool of the runner, a finite-population factor makes this value a little smaller. The grid ignores this factor, so it is conservative.

The second table uses the `sd_d` measured under the synthetic degradation (the mean over 5 draws of random documents for all queries in the pool). It compares the analytic MDE with the **simulated** power of the harmful rule (`n_boot = 2000`, as in the runner), for both populations. The degradation can assign damage in two ways, and they are not interchangeable:
- **Fixed fraction:** exactly `s * 100` queries in each window (the power table in section 6).
- **Bernoulli:** each query is damaged independently with probability `s`. This adds variation in the number of damaged queries, and the analytic `sd_d` describes this case.

The table shows both simulated powers, so that the analytic and simulated values are compared for the same case.

**Why this test**
- **Estimand:** the smallest mean nDCG drop that the harmful rule detects 80% of the time at w = 100.
- **Unit of independence:** the query in a window.
- **Assumptions:** the analytic formula assumes an approximately normal mean difference. nDCG differences under all-or-nothing damage are skewed. Thus, the simulation is the reference, and the formula is a check.
- **Why not the usual alternative:** a simulation alone gives no rule of thumb for an unknown failure. The formula gives one, after this section checks its accuracy.

In [ ]:
z_ = stats.norm.ppf(0.95) + stats.norm.ppf(0.80)
grid = pd.DataFrame({"sd_d": [0.1, 0.2, 0.3, 0.4]})
for w in (50, 100, 200, 400):
    grid[f"MDE_w{w}"] = z_ * grid["sd_d"] / np.sqrt(w)
display(grid.round(3))

rows = []
for pop in POPS:
    _, pool = ref_pool(pop, 0)
    rng = np.random.default_rng([SEED, 7])
    rnd_draws = np.array([random_lists(pool, rng)[1] for _ in range(5)])        # Shape (5, pool): nDCG under random documents
    for s in (0.05, 0.1, 0.2, 0.3, 0.5):
        diffs = [clean_nd[pool] - np.where(rng.random(len(pool)) < s, rd, clean_nd[pool]) for rd in rnd_draws]
        mean_drop, sd_d = np.mean([d.mean() for d in diffs]), np.mean([d.std() for d in diffs])
        hits = 0
        for m in range(300):
            rt = np.random.default_rng([SEED, 7, m])
            win = rt.choice(pool, W, replace=False)
            _, rnd = random_lists(win, rt)
            a = rt.random(W) < s                                   # Bernoulli: each query is damaged with probability s
            hits += paired_bootstrap(clean_nd[win], np.where(a, rnd, clean_nd[win]), n_boot=2000, seed=m)[1] < 0.05
        rows.append({"population": pop, "severity": s, "true_mean_drop": mean_drop, "sd_d_bernoulli": sd_d,
                     "analytic_MDE_w100": z_ * sd_d / np.sqrt(W),
                     "analytic_power_w100": float(stats.norm.cdf(mean_drop / (sd_d / np.sqrt(W)) - stats.norm.ppf(0.95))),
                     "simulated_power_bernoulli": hits / 300,
                     "simulated_power_fixed_fraction": float(by_ref[(by_ref["population"] == pop) & (by_ref["severity"] == s)]["harmful_rate"].mean())})
pd.DataFrame(rows).round(3)

In [ ]:
import datasets, matplotlib, sentence_transformers, sklearn, scipy, torch
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "matplotlib": matplotlib.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
       "datasets": datasets.__version__, "seed": SEED, "device": "cpu", "window_size": W,
       "synthetic_degradation": "random retrieval for a fraction of queries"})

## Findings
All numbers come from one full run (seed 0, NFCorpus, window size 100). **Runner** = the 323 test queries that the runner uses (reference 100, pool 223). **All** = all 3,237 labelled queries (sensitivity).

- **Label-free detectors need much more damage than the ground-truth rule.** On the runner population, the median earliest sustained 80% detection is **0.30 for nominal KS** and **0.25 for nominal Welch z** (range 0.2 to 0.4 across references). Only `harmful_rate` reaches 80% at 0.10. But `harmful_rate` is the qrels-based ground truth, not a detector. The placeholder damage replaces the full top-10 with random documents. Real F1 to F5 damage is smaller, so these detectors will probably need more damage than this.
- **Power under the placeholder degradation** (runner population, 30 references, mean detection rate):

  | Rule | Severity 0.05 | Severity 0.10 | Median earliest 80% (range) |
  |---|---|---|---|
  | Harmful rule (ground truth, uses qrels) | 0.67 | 0.99 | 0.1 (0.1 to 0.1) |
  | KS, nominal | 0.01 | 0.05 | 0.3 (0.2 to 0.4) |
  | Welch z, nominal | 0.03 | 0.13 | 0.25 (0.2 to 0.4) |
  | KS, oracle-calibrated (upper limit) | 0.09 | 0.24 | 0.2 (0.2 to 0.4) |
  | Welch z, oracle-calibrated (upper limit) | 0.16 | 0.44 | 0.2 (0.1 to 0.4) |

  The oracle thresholds use clean pool windows that a `Detector` does not see. The reference-only calibration, which a detector can calculate in `fit()`, is **not calibrated**: at severity 0, it alarms 22% (KS) and 10% (z) of the time on the runner population, and 37% / 22% on all queries.
- **Severity is not the fraction of queries with a real drop.** 33% of queries (1,070 of 3,237) already have recall@10 = 0 on the clean system, so their nDCG cannot decrease. Thus, at severity 0.3, only approximately 20% of the window has a real drop.
- **The harmful label itself is not stable at low damage.** At severity 0.05, 32% of the windows of the runner get a different call under different bootstrap seeds. This share is the same at `n_boot` = 500, 2,000, and 10,000. More resamples also change the label rate: the mean harmful rate is 0.654, 0.686, and 0.729. The cause is a lattice:
  - With sparse damage, the bootstrap p is approximately `(1 - j/100)^100`. j is the number of damaged queries whose nDCG decreased.
  - For j = 3, this is 0.0476, just below 0.05.
  - Windows with j = 3 are seed-dependent 97% of the time. Windows with other values of j are seed-dependent 0% of the time.

  Thus, the instability is in the ground-truth label, not only in the detectors. Real injectors at low severity also give sparse differences. Harmful labels near this limit are partly Monte Carlo noise, fixed by `seed=trial`.
- **Window noise agrees with the finite-population formula** in both populations (ratio 0.98 to 1.02). At w = 100, the window SD is **0.022 (runner)** and 0.029 (all). The iid formula without the correction gives 0.030 for the runner, which is a third too large. Window means are approximately normal at w ≤ 100 (Shapiro p 0.37 to 0.88). At w = 200, the runner pool is almost empty, and normality fails (p = 0.003).
- **The harmful rule is approximately calibrated, but not exactly.** Under a symmetric sign-flip null, its false-positive rate is **7.1% (5.6 to 8.8%) for the runner** and 5.0% (3.8 to 6.5%) for all queries. The runner interval is too narrow, because its windows overlap.
- **Detector false alarms depend on the reference.** Mean false-alarm rates at a nominal 5%, top-1 score, runner population:

  | Rule | Mean false alarm | SD across references | Max |
  |---|---|---|---|
  | KS | 2.9% | 0.08 | 0.49 |
  | Welch z | 5.1% | 0.13 | 0.81 |
  | Naive z (bug) | 16.4% | – | – |

  On the runner's own reference, KS and Welch z give no false alarms in 300 clean windows.
- **The top-1 score separates failing queries.** AUROC 0.79 (0.74 to 0.84) on the queries of the runner, and 0.785 (0.77 to 0.80) on all queries. A combination of the three score signals adds nothing.
- **Window-level tracking (H3, all queries).** The mean window-level Spearman between top-1 and nDCG is 0.46 at w = 100. The matched null is 0.00 ± 0.02, and p = 0.0033 is the lowest possible value for 300 permutations.
- **Minimum detectable drop** for a 100-query window with a per-query SD of 0.2: approximately 0.05 nDCG. At severity 0.05, the analytic power (0.48) is lower than the simulated power (0.58 Bernoulli, 0.67 fixed fraction), because the nDCG differences are skewed.

### What this supports
For the setting of the runner (323 NFCorpus test queries, a 100-query reference, windows of 100 from a 223-query pool, clean system):
- Window-mean nDCG@10 has a known noise floor of approximately 0.022.
- The paired-bootstrap harmful rule is approximately calibrated (5 to 7% under a symmetric null). But its label is not stable for windows with sparse damage near p = 0.05.
- A Welch z or KS detector with a 100-query reference is much less sensitive than the ground-truth rule. With nominal thresholds, it needs a median severity of 0.25 to 0.30 for sustained 80% detection under the placeholder damage. Its false-alarm rate depends strongly on the reference.

### What this does not support
- That detectors need only approximately 10% to 20% damaged queries. Only the ground-truth rule reaches 80% at 0.10.
- How detectable **F1 to F5** are. The degradation here is a placeholder: damaged queries lose all of their nDCG. This is much easier to see than the smaller damage of real injectors.
- That the calibrated detector power is achievable. The oracle thresholds use clean pool windows that the `Detector` interface does not see, and the achievable reference-only calibration gives too many alarms.
- That the trials of the runner are independent replicates. Windows from a 223-query pool share approximately 45 queries for each pair.
- That harmful labels near the p = 0.05 limit are stable. At low damage, they can change with the bootstrap seed and with `n_boot` (section 6b).
- That the top-1 values are out-of-sample. If top-1 is selected because of its correlation on these queries, the values are in-sample for that selection. Also, the hypotheses were not formed blind to the held-out datasets (see the README).
- That the noise floor applies to **production traffic**, which has topic clusters, or that the thresholds transfer to a different corpus, model, or window size.